In [0]:
WITH params AS (
  SELECT 30 AS p_window_days
),
normalized AS (
  SELECT
    try_to_timestamp(ingested_at_utc) AS ingested_ts_utc,
    CASE
      WHEN upper(trim(COALESCE(title, ''))) LIKE 'DEPRECATED:%' THEN 1
      ELSE 0
    END AS is_deprecated
  FROM 3dt2ndteam5.cwe.cwe_weaknesses
),
scoped AS (
  SELECT
    ingested_ts_utc,
    is_deprecated
  FROM normalized
  CROSS JOIN params p
  WHERE ingested_ts_utc IS NOT NULL
    AND ingested_ts_utc >= timestampadd(DAY, -p.p_window_days, current_timestamp())
),
daily AS (
  SELECT
    date(ingested_ts_utc) AS metric_date_utc,
    date(from_utc_timestamp(ingested_ts_utc, 'Asia/Seoul')) AS metric_date_kst,
    COUNT(*) AS total_rows,
    SUM(is_deprecated) AS deprecated_rows
  FROM scoped
  GROUP BY
    date(ingested_ts_utc),
    date(from_utc_timestamp(ingested_ts_utc, 'Asia/Seoul'))
)
SELECT
  metric_date_utc,
  metric_date_kst,
  total_rows,
  deprecated_rows,
  ROUND(
    CASE
      WHEN total_rows = 0 THEN 0.0
      ELSE (deprecated_rows * 100.0) / total_rows
    END,
    2
  ) AS deprecated_ratio_pct
FROM daily
ORDER BY metric_date_utc;
